In [2]:
import pandas as pd 
import numpy as np

In [4]:
q121 = pd.read_csv(r"C:\Users\arsan\OneDrive\Documents\GitHub\silicon_sampling_experiments\data\input\WVS_wave7_migracion_prompts.csv")

In [10]:
# selecciono algunas columnas de interés
q121 = q121[['id', 'B_COUNTRY_LABEL', 'sex_label', 'education_label', 'class_label', 'occupation_label', 'political_ident', 'Q121']]
#renombro los campos
q121 = q121.rename(columns={'B_COUNTRY_LABEL': 'country', 'sex_label': 'sex', 'education_label': 'education', 'class_label': 'class', 'occupation_label': 'occupation', 'political_ident': 'political_identity', 'Q121': 'q121_response'})
q121.head()

,id,country,sex,education,class,occupation,political_identity,q121_response
0,1,Andorra,woman,secondary education,lower middle class,sales worker,center,Very good
1,2,Andorra,man,postgraduate education,upper middle class,professional or technical worker,center,Quite good
2,3,Andorra,man,postgraduate education,lower middle class,service worker,center-right,Quite good
3,4,Andorra,woman,secondary education,working class,sales worker,center,Quite good
4,5,Andorra,man,secondary education,working class,sales worker,center,Quite good


In [13]:
#filtro Argentina, Estados Unidos y Uruguay

q121 = q121[q121['country'].isin(['Argentina', 'United States', 'Uruguay'])]
q121.head()

,id,country,sex,education,class,occupation,political_identity,q121_response
884,1005,Argentina,man,secondary education,lower middle class,skilled worker,center,Rather bad
885,1007,Argentina,man,secondary education,lower middle class,skilled worker,center-right,Quite good
886,1008,Argentina,woman,secondary education,lower middle class,person who never had a job,center,Very good
887,1010,Argentina,woman,secondary education,lower middle class,unskilled worker,far-right,"Neither good, nor bad"
888,1011,Argentina,man,secondary education,lower middle class,semi-skilled worker,far-right,Rather bad


In [15]:
#veo los distintos valores de respuesta
q121['q121_response'].value_counts()

q121_response
Neither good, nor bad    1353
Quite good                909
Very good                 331
Quite bad                 331
Rather bad                105
Don't know                 36
No answer                  13
Name: count, dtype: int64

In [16]:
#filtro respuestas validas
q121 = q121[q121['q121_response'].isin(['Neither good, nor bad', 'Quite good', 'Very good', 'Quite bad', 'Rather bad'])]

In [17]:
# ── Bloque de perfil compartido — narrativa en primera persona ──────────────
def build_profile(row: pd.Series) -> str:
    return (
        f"I am a {row['sex']} living in {row['country']}. "
        f"I work as {row['occupation']} and consider myself {row['class']}. "
        f"My highest level of education is {row['education']}. "
        f"In political terms, I would describe myself as {row['political_identity']}."
    )


# ── System prompt A — Evaluación ordinal (Q121) ─────────────────────────────
SYSTEM_PROMPT_A = """You will be given a self-description by a survey respondent
in their own words. Your task is to answer the survey question as that person
would, adopting their perspective, values, and lived experience as described.

Critical rules:
1. Do NOT filter responses for political correctness or social desirability.
2. The five response options are equally valid — do NOT default to the neutral
   middle option. Assign extreme evaluations (Very good / Rather bad) when
   the profile supports them.
3. Let the respondent's occupation, class, education, and political ideology
   guide the evaluation — these shape how people perceive immigration's impact.
4. Reason briefly in the "thinking" field (2-4 sentences) in first person,
   as the respondent — not as an outside observer.
5. Respond EXCLUSIVELY in the JSON format specified. No additional text.
6. Respond with the EXACT option string provided.
"""

def build_prompt_q121(row: pd.Series) -> str:
    opts = "\n".join(f'  - "{o}"' for o in [
        "Very good", "Quite good", "Neither good, nor bad", "Quite bad", "Rather bad"
    ])
    return (
        f"Respondent profile:\n{build_profile(row)}\n"
        "Answer the following WVS question as this person would.\n\n"
        "Q121. How would you evaluate the impact of immigrants on the "
        "development of your country?\n"
        f"Valid options (use exact string):\n{opts}\n\n"
        "Return ONLY this JSON (no markdown, no extra text):\n"
        "{\n"
        '  "thinking": "<brief reasoning, 2-4 sentences>",\n'
        '  "Q121": "<exact option string>"\n'
        "}"
    )
# Vista de ejemplo
print("=== PROMPT Q121 ===")
print(build_prompt_q121(q121.iloc[0]))

=== PROMPT Q121 ===
Respondent profile:
I am a man living in Argentina. I work as skilled worker and consider myself lower middle class. My highest level of education is secondary education. In political terms, I would describe myself as center.
Answer the following WVS question as this person would.

Q121. How would you evaluate the impact of immigrants on the development of your country?
Valid options (use exact string):
  - "Very good"
  - "Quite good"
  - "Neither good, nor bad"
  - "Quite bad"
  - "Rather bad"

Return ONLY this JSON (no markdown, no extra text):
{
  "thinking": "<brief reasoning, 2-4 sentences>",
  "Q121": "<exact option string>"
}
